# **Machine Learning Pipeline: Heart Failure Prediction**

## **Setup & Import Library**

In [ ]:
import os
import sys
import tensorflow as tf
import tensorflow_model_analysis as tfma
from tfx.components import (
    CsvExampleGen,
    StatisticsGen,
    SchemaGen,
    ExampleValidator,
    Transform,
    Tuner,
    Trainer,
    Evaluator,
    Pusher,
)
from tfx.proto import trainer_pb2, example_gen_pb2, pusher_pb2
from tfx.orchestration.experimental.interactive.interactive_context import (
    InteractiveContext,
)
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import (
    LatestBlessedModelStrategy,
)
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

# Menambahkan root folder path
sys.path.append("..")

## **Inisialisasi**

In [ ]:
PIPELINE_NAME = "fikri_rouzan-pipeline"
PIPELINE_ROOT = os.path.join("..", PIPELINE_NAME)

# Seluruh artifact TFX akan disimpan di folder
context = InteractiveContext(pipeline_name=PIPELINE_NAME, pipeline_root=PIPELINE_ROOT)

## **Data Ingestion**

In [ ]:
DATA_ROOT = os.path.join("..", "data")

# Membagi dataset
output_config = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(
        splits=[
            example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
            example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2),
        ]
    )
)

example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output_config)
context.run(example_gen)

## **Data Validation**

In [ ]:
# Menghitung statistik deskriptif dataset
statistics_gen = StatisticsGen(examples=example_gen.outputs["examples"])
context.run(statistics_gen)
context.show(statistics_gen.outputs["statistics"])

# Generate skema tipe data dan rentang nilai
schema_gen = SchemaGen(statistics=statistics_gen.outputs["statistics"])
context.run(schema_gen)
context.show(schema_gen.outputs["schema"])

# Memeriksa anomali data berdasarkan skema
example_validator = ExampleValidator(
    statistics=statistics_gen.outputs["statistics"], schema=schema_gen.outputs["schema"]
)
context.run(example_validator)
context.show(example_validator.outputs["anomalies"])

## **Feature Engineering**

In [ ]:
TRANSFORM_MODULE_FILE = os.path.join("..", "modules", "transform.py")

transform = Transform(
    examples=example_gen.outputs["examples"],
    schema=schema_gen.outputs["schema"],
    module_file=TRANSFORM_MODULE_FILE,
)
context.run(transform)

## **Hyperparameter Tuning**

In [ ]:
TUNER_MODULE_FILE = os.path.join("..", "modules", "tuner.py")

tuner = Tuner(
    module_file=TUNER_MODULE_FILE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=10),
)
context.run(tuner)

## **Model Training**

In [ ]:
TRAINER_MODULE_FILE = os.path.join("..", "modules", "trainer.py")

trainer = Trainer(
    module_file=TRAINER_MODULE_FILE,
    examples=transform.outputs["transformed_examples"],
    transform_graph=transform.outputs["transform_graph"],
    schema=schema_gen.outputs["schema"],
    hyperparameters=tuner.outputs["best_hyperparameters"],
    train_args=trainer_pb2.TrainArgs(splits=["train"], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=["eval"], num_steps=10),
)
context.run(trainer)

## **Model Resolver & Evaluator**

In [ ]:
# Mendapatkan model terbaik sebelumnya untuk dijadikan baseline
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing),
).with_id("latest_blessed_model_resolver")

context.run(model_resolver)

# Menguji performa model dengan threshold akurasi minimal 60%
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key="HeartDisease_xf")],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(
            metrics=[
                tfma.MetricConfig(class_name="BinaryAccuracy"),
                tfma.MetricConfig(class_name="ExampleCount"),
                tfma.MetricConfig(class_name="AUC"),
                tfma.MetricConfig(
                    class_name="BinaryAccuracy",
                    threshold=tfma.MetricThreshold(
                        value_threshold=tfma.GenericValueThreshold(
                            lower_bound={"value": 0.6}
                        ),
                        change_threshold=tfma.GenericChangeThreshold(
                            direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                            absolute={"value": -1e-10},
                        ),
                    ),
                ),
            ]
        )
    ],
)

evaluator = Evaluator(
    examples=example_gen.outputs["examples"],
    model=trainer.outputs["model"],
    baseline_model=model_resolver.outputs["model"],
    eval_config=eval_config,
)
context.run(evaluator)

## **Model Deployment Artifact**

In [ ]:
SERVING_MODEL_DIR = os.path.join("..", "serving_model", "heart-failure-model")

pusher = Pusher(
    model=trainer.outputs["model"],
    model_blessing=evaluator.outputs["blessing"],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory=SERVING_MODEL_DIR
        )
    ),
)
context.run(pusher)